In [1]:
import numpy as np
import xarray as xr

In [2]:
def parse_dataset(path:str):
    _data = np.loadtxt(path, unpack=True)
    wave_number, _counts = _data[-2], _data[-1]
    num_coords = len(_data) - 2
    coords = [(_data[idx]-_data[idx].min()).astype(int) for idx in range(num_coords)]
    unique_coords = [np.unique(coord, return_inverse=True) for coord in coords]
    unique_coords += [np.unique(wave_number, return_inverse=True)] # [(X_value, X_idx), ... (WN_value, WN_idx)]
    idxs, values = [coord[1] for coord in unique_coords], [coord[0] for coord in unique_coords]
    counts_shape = tuple([len(coord[0]) for coord in unique_coords])
    counts = np.zeros(counts_shape, dtype = _counts.dtype)

    for z in zip(*idxs, _counts):
        counts[*z[:-1]] = z[-1]

    dimension_names = [f'X_{i}' for i in range(len(values) -1)] + ['wave_number']

    counts = xr.DataArray(counts, coords=values, dims=dimension_names)
    counts.wave_number.attrs['units'] = 'cm^-1'
    return counts

In [4]:
parse_raw = True
data_files = ['raw_data/Mixed_8x8.txt', 'raw_data/map_c_100x10.txt', 'raw_data/map_c_50x20_mixed2.txt', 'raw_data/DepthMapHR100x5p1s1acc.txt']
system_type = ['SiC+Graphene', 'SiC', 'SiC+Graphene', 'SiC']
if parse_raw:
    datasets = [parse_dataset(path) for path in data_files]
    for i, dataset in enumerate(datasets):
        dataset.attrs['system_type'] = system_type[i]
        #Save to netcdf
        dims = dataset.dims[:-1]
        fname = f'../data/{system_type[i]}_'
        for dim_idx in range(len(dims) -1):
            fname += f'{len(dataset[dims[dim_idx]])}x'
        fname += f'{len(dataset[dims[-1]])}.nc'
        dataset.to_netcdf(fname)